# Fine-tuning LLaMA 3.1 8B với Unsloth và LoRA

Tệp notebook này chứa quy trình tinh chỉnh (fine-tuning) mô hình ngôn ngữ LLaMA 3.1 8B bằng kỹ thuật LoRA và thư viện Unsloth. Dữ liệu huấn luyện là tập dữ liệu hỏi đáp y tế tiếng Việt.


In [1]:
!pip install -qqq "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --progress-bar off
!pip install -qqq --no-deps "xformers<0.0.24" "trl<0.9.0" peft accelerate bitsandbytes "wandb" --progress-bar off

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 1.42.0 requires rich<14,>=12.4.4, but you have rich 14.0.0 which is incompatible.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [2]:
import os
import torch
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
import psutil # Để lấy số CPU cores

Unsloth: Patching Xformers to fix some performance issues.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-05-27 18:21:07.322839: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748370067.527519      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748370067.587121      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# --- 1. KIỂM TRA GPU VÀ CẤU HÌNH UNLOTH ---
major_version, minor_version = torch.cuda.get_device_capability()
print(f"GPU Capability: {major_version}.{minor_version}")


# --- 2. THAM SỐ CƠ BẢN ---
MAX_SEQ_LENGTH = 2048 
MODEL_NAME = "unsloth/llama-3.1-8b-Instruct-bnb-4bit" 
OUTPUT_DIR = "llama3_1_8b_unsloth_finetuned_vi"

GPU Capability: 7.5


In [4]:
# --- 3. TẢI DỮ LIỆU GỐC ---
data_files = {
    "train": "/kaggle/input/fine-tune-data/train.jsonl", 
    "val": "/kaggle/input/fine-tune-data/val.jsonl", 
}
raw_dataset = load_dataset("json", data_files=data_files)
print("Số lượng mẫu ban đầu:")
print(f"Train: {len(raw_dataset['train'])}")
print(f"Validation: {len(raw_dataset['val'])}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Số lượng mẫu ban đầu:
Train: 5974
Validation: 750


In [ ]:
# --- 4. TẢI MODEL VÀ TOKENIZER (CẦN TOKENIZER ĐỂ LỌC) ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH, 
    dtype = None,
    load_in_4bit = True,
)

# Kiểm tra và đặt pad_token nếu cần
if tokenizer.pad_token is None:
    print("Tokenizer không có pad_token. Đang đặt pad_token thành eos_token.")
    tokenizer.pad_token = tokenizer.eos_token
    

==((====))==  Unsloth 2025.5.7: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [6]:
def format_and_filter_prompt_llama3_1_vi(example, tokenizer, max_length):
    system_prompt = "Bạn là một chuyên gia về y tế và sức khỏe. Hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi."
    user_message = f"Ngữ cảnh: {example['context']}\nCâu hỏi: {example['question']}"
    assistant_message = example['answer']

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": assistant_message},
    ]

    # add_generation_prompt=False vì đây là dữ liệu training
    # tokenize=True để đếm số token
    tokenized_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt" # Để lấy input_ids
    )

    num_tokens = tokenized_prompt.input_ids.shape[1]

    if num_tokens <= max_length:
        # Nếu số token hợp lệ, trả về prompt dạng text để SFTTrainer xử lý sau
        # (SFTTrainer với packing=True sẽ tự tokenize lại)
        formatted_text_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": formatted_text_prompt, "num_tokens": num_tokens, "keep": True}
    else:
        return {"text": None, "num_tokens": num_tokens, "keep": False}

In [7]:
print(f"\nĐang lọc dữ liệu để giữ lại các mẫu có số token <= {MAX_SEQ_LENGTH}...")

from functools import partial


filtered_dataset = {}
for split in raw_dataset.keys():
    print(f"Đang xử lý tập {split}...")
    # Tăng num_proc để tăng tốc, nhưng cẩn thận với tài nguyên Kaggle
    # Kaggle notebook thường có 2 hoặc 4 CPU cores.
    num_map_proc = min(4, psutil.cpu_count(logical=False) or 1) # Giới hạn tối đa 4 cores cho map

    def format_and_filter_prompt_llama3_1_vi_batched(examples, tokenizer, max_length):
        texts = []
        num_tokens_list = []
        keep_flags = []

        system_prompt_template = "Bạn là một chuyên gia về y tế và sức khỏe. Hãy sử dụng ngữ cảnh được cung cấp để trả lời câu hỏi."

        for i in range(len(examples['question'])):
            user_message = f"Ngữ cảnh: {examples['context'][i]}\nCâu hỏi: {examples['question'][i]}"
            assistant_message = examples['answer'][i]

            messages = [
                {"role": "system", "content": system_prompt_template},
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": assistant_message},
            ]

            tokenized_prompt_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False
            ) # trả về list of token ids
            num_tokens = len(tokenized_prompt_ids)

            if num_tokens <= max_length:
                formatted_text_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=False
                )
                texts.append(formatted_text_prompt)
                keep_flags.append(True)
            else:
                texts.append(None) # Hoặc một giá trị placeholder
                keep_flags.append(False)
            num_tokens_list.append(num_tokens)

        return {"text": texts, "num_tokens": num_tokens_list, "keep": keep_flags}


    # Áp dụng hàm map đã sửa cho batch
    processed_split_ds = raw_dataset[split].map(
        partial(format_and_filter_prompt_llama3_1_vi_batched, tokenizer=tokenizer, max_length=MAX_SEQ_LENGTH),
        batched=True,
        batch_size=100, # Điều chỉnh batch_size cho map
        num_proc=num_map_proc,
        remove_columns=[col for col in raw_dataset[split].column_names if col not in ["text", "keep"]] # Giữ lại cột text và keep
    )

    # Lọc dựa trên cờ 'keep'
    filtered_dataset[split] = processed_split_ds.filter(lambda example: example['keep'], num_proc=num_map_proc)
    # Loại bỏ các cột không cần thiết sau khi lọc
    filtered_dataset[split] = filtered_dataset[split].remove_columns(['keep', 'num_tokens'])


print("\nSố lượng mẫu sau khi lọc:")
train_dataset_final = filtered_dataset["train"]
val_dataset_final = filtered_dataset["val"]
print(f"Train: {len(train_dataset_final)}")
print(f"Validation: {len(val_dataset_final)}")

if len(train_dataset_final) == 0:
    raise ValueError("Không còn mẫu nào trong tập train sau khi lọc. Hãy kiểm tra MAX_SEQ_LENGTH hoặc dữ liệu của bạn.")


Đang lọc dữ liệu để giữ lại các mẫu có số token <= 2048...
Đang xử lý tập train...


Map (num_proc=2):   0%|          | 0/5974 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/5974 [00:00<?, ? examples/s]

Đang xử lý tập val...


Map (num_proc=2):   0%|          | 0/750 [00:00<?, ? examples/s]

Filter (num_proc=2):   0%|          | 0/750 [00:00<?, ? examples/s]


Số lượng mẫu sau khi lọc:
Train: 5751
Validation: 713


In [8]:
# --- 6. CẤU HÌNH LoRA ---
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
)
model.print_trainable_parameters()


Unsloth 2025.5.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
# --- 7. THIẾT LẬP TRAINING ARGUMENTS --- (không cần thiết vì đã thiết lập ở cell dưới)
num_gpus = torch.cuda.device_count()
per_device_bs = 1 
grad_accum_steps = 16

num_train_epochs = 1 # Bắt đầu với 1 epoch.


if len(train_dataset_final) < (per_device_bs * num_gpus * grad_accum_steps):
    print(f"Cảnh báo: Số lượng mẫu train ({len(train_dataset_final)}) nhỏ hơn effective batch size ({per_device_bs * num_gpus * grad_accum_steps}).")
    print("Điều chỉnh gradient_accumulation_steps để đảm bảo ít nhất 1 step mỗi epoch.")
    # Đảm bảo grad_accum_steps ít nhất là 1
    grad_accum_steps = max(1, len(train_dataset_final) // (per_device_bs * num_gpus) or 1)
    print(f"gradient_accumulation_steps được cập nhật thành: {grad_accum_steps}")


total_train_steps_approx = (len(train_dataset_final) / (per_device_bs * num_gpus * grad_accum_steps)) * num_train_epochs
warmup_steps = max(10, int(0.1 * total_train_steps_approx)) # 10% tổng số bước, tối thiểu 10
save_and_eval_steps = max(25, int(0.25 * total_train_steps_approx)) # Lưu và eval mỗi 25% epoch, tối thiểu 25

# Kiểm tra nếu total_train_steps_approx quá nhỏ (ví dụ < save_and_eval_steps)
if total_train_steps_approx < save_and_eval_steps and total_train_steps_approx > 0 :
    save_and_eval_steps = int(total_train_steps_approx) # Eval ít nhất 1 lần nếu có thể
elif total_train_steps_approx == 0: # Không có đủ dữ liệu cho 1 step
    save_and_eval_steps = 1
    warmup_steps = 0
    print("Cảnh báo: Không đủ dữ liệu cho ít nhất một training step với cấu hình hiện tại.")

In [10]:
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_api_key)
except:
    wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tralalelo (tralalelo-no) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
training_args = TrainingArguments(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 1,
    per_device_eval_batch_size = 2,
    gradient_accumulation_steps = 16,
    num_train_epochs = 1,
    learning_rate = 2e-4, 
    fp16 = True,
    bf16 = False,
    logging_strategy = "steps",
    logging_steps = 1, 
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    warmup_steps = 10,
    save_strategy = "epoch",
    save_total_limit = 1,
    eval_strategy = "steps",
    eval_steps = 18,
    dataloader_num_workers = min(4, psutil.cpu_count(logical=False) or 1),
    seed = 42,
    run_name = "llama3.1-8b-unsloth-medical-vi-3",
    report_to = "wandb",
)

In [12]:
# --- 8. KHỞI TẠO SFTTRAINER ---
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset_final,
    eval_dataset = val_dataset_final,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    args = training_args,
    packing = True, 
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [13]:
if len(train_dataset_final) > 0 and len(val_dataset_final) > 0:
    print("Bắt đầu huấn luyện...")
    trainer_stats = trainer.train()
    print("Huấn luyện hoàn tất.")

    final_model_path = os.path.join(OUTPUT_DIR, "final_checkpoint")
    model.save_pretrained(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    print(f"Adapter LoRA và tokenizer đã được lưu vào {final_model_path}")

else:
    print("Không đủ dữ liệu trong tập train hoặc validation để bắt đầu huấn luyện sau khi lọc.")
    if len(train_dataset_final) == 0: print("Tập train rỗng.")
    if len(val_dataset_final) == 0: print("Tập validation rỗng (có thể chấp nhận được nếu bạn chỉ muốn train).")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,966 | Num Epochs = 1 | Total steps = 92
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Bắt đầu huấn luyện...


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
18,0.028800,0.025404
36,0.005200,0.004933
54,0.001400,0.001426
72,0.001100,0.001316
90,0.001200,0.001278


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Huấn luyện hoàn tất.
Adapter LoRA và tokenizer đã được lưu vào llama3_1_8b_unsloth_finetuned_vi/final_checkpoint


In [18]:
!pip install huggingface_hub -q


In [19]:
from huggingface_hub import login, upload_folder, HfApi
from kaggle_secrets import UserSecretsClient
from peft import PeftModel

# Lấy token từ Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("1HF_TOKEN")

# Đăng nhập
login(token=hf_token)



In [21]:
from huggingface_hub import create_repo
repo_id = "tralalelo/fine-tune-Llama3.1-8B-adapter-28-5"
create_repo(repo_id, exist_ok=True)

RepoUrl('https://huggingface.co/tralalelo/fine-tune-Llama3.1-8B-adapter-28-5', endpoint='https://huggingface.co', repo_type='model', repo_id='tralalelo/fine-tune-Llama3.1-8B-adapter-28-5')

In [22]:
# Đẩy adapter LoRA và tokenizer lên HF
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)
print(f"✅ Đã đẩy mô hình LoRA lên: https://huggingface.co/{repo_id}")

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Saved model to https://huggingface.co/tralalelo/fine-tune-Llama3.1-8B-adapter-28-5


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

✅ Đã đẩy mô hình LoRA lên: https://huggingface.co/tralalelo/fine-tune-Llama3.1-8B-adapter-28-5


In [24]:
model.push_to_hub_merged("tralalelo/fine-tune-Llama3.1-8B-merged-4bit-28-5", tokenizer, save_method="merged_4bit_forced")

Unsloth: Merging 4bit and LoRA weights to 4bit...
This might take 5 minutes...


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:355: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Done.
Unsloth: Saving 4bit Bitsandbytes model. Please wait...


README.md:   0%|          | 0.00/598 [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.65G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/604 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Saved merged_4bit model to https://huggingface.co/tralalelo/fine-tune-Llama3.1-8B-merged-4bit-28-5


In [25]:
model.push_to_hub_gguf("tralalelo/FineLlama-3.1-8B-GGUF", tokenizer, "q4_k_m")

Cloning into 'llama.cpp'...
Submodule 'kompute' (https://github.com/nomic-ai/kompute.git) registered for path 'ggml/src/ggml-kompute/kompute'
Cloning into '/kaggle/working/llama.cpp/ggml/src/ggml-kompute/kompute'...
Submodule path 'ggml/src/ggml-kompute/kompute': checked out '4565194ed7c32d1d2efa32ceab4d3c6cae006306'
make: Entering directory '/kaggle/working/llama.cpp'
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.34.1")
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAV

Unsloth: You have 2 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 5.7G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 19.42 out of 31.35 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


  0%|          | 0/32 [00:00<?, ?it/s]
We will save to Disk and not RAM now.
 38%|███▊      | 12/32 [00:06<00:10,  1.85it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 118.12 MiB is free. Process 3703 has 14.62 GiB memory in use. Of the allocated memory 14.37 GiB is allocated by PyTorch, and 94.61 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)